<a href="https://colab.research.google.com/github/ArunaDevi28/Machine-Learning-Project/blob/main/MLPAD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ======= Install dependencies (run once in Colab) =======
!pip install -q transformers torch torchvision pillow gradio gTTS

# Optional: if you want faster translation you can pip install "sentencepiece" too:
!pip install -q sentencepiece

# Uninstall and then install click>=8.2.0 to resolve dependency conflict with wandb
!pip uninstall -y click
!pip install -q click==8.2.0 --upgrade --force-reinstall

# ======= Imports =======
import os, json, uuid
from PIL import Image
import gradio as gr
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
from transformers import MarianMTModel, MarianTokenizer
from gtts import gTTS
from difflib import SequenceMatcher
from datetime import datetime

# ======= Device =======
device = "cuda" if torch.cuda.is_available() else "cpu"

# ======= Load BLIP model (this may take time first run) =======
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

# ======= Helper: Translation (Helsinki opus-mt) =======
# Cache tokenizer+model per target language in a dict to avoid reloading repeatedly
_translation_cache = {}
def translate_text(text, tgt_lang):
    """Translate English text to target language code (e.g., 'fr','hi','es')."""
    if tgt_lang == "en" or not text:
        return text
    model_name = f"Helsinki-NLP/opus-mt-en-{tgt_lang}"
    if model_name not in _translation_cache:
        tokenizer = MarianTokenizer.from_pretrained(model_name)
        m = MarianMTModel.from_pretrained(model_name).to(device)
        _translation_cache[model_name] = (tokenizer, m)
    tokenizer, m = _translation_cache[model_name]
    batch = tokenizer([text], return_tensors="pt", padding=True).to(device)
    gen = m.generate(**batch, max_length=80)
    out = tokenizer.decode(gen[0], skip_special_tokens=True)
    return out

# ======= Caption Quality/Diversity heuristic =======
def diversity_score(captions):
    """Simple diversity score: average pairwise edit distance normalized."""
    n = len(captions)
    if n < 2: return 0.0
    total = 0.0
    pairs = 0
    for i in range(n):
        for j in range(i+1, n):
            a, b = captions[i], captions[j]
            # similarity ratio (higher = more similar)
            sim = SequenceMatcher(None, a, b).ratio()
            total += (1 - sim)  # diversity = 1 - similarity
            pairs += 1
    return round((total / pairs) * 100, 1)  # percent

# ======= Core: Generate captions =======
def gen_captions_and_plain(image, style, creativity, max_len, num_captions=3):
    """
    Generate captions using BLIP with sampling + beam control.
    style: one of styles (affects post-processing)
    creativity: temperature float (0.1 - 1.5)
    max_len: max tokens (10-60)
    """
    if image is None:
        return "<div style='color:#a00'>No image uploaded.</div>", "", 0.0

    # Prepare inputs
    inputs = processor(images=image, return_tensors="pt").to(device)

    # generate multiple sequences in one pass
    outputs = model.generate(
        **inputs,
        max_length=int(max_len),
        num_return_sequences=num_captions,
        do_sample=True,
        temperature=float(creativity),
        top_k=50,
        top_p=0.95,
        num_beams=5,
        early_stopping=True
    )

    captions = [processor.decode(o, skip_special_tokens=True).strip() for o in outputs]

    # Style transforms
    styled = []
    for cap in captions:
        if style == "Short & Direct":
            s = cap.split(",")[0].split(" and ")[0].strip()
        elif style == "Fun & Creative":
            s = f"✨ {cap.capitalize()} ✨"
        elif style == "Descriptive & Detailed":
            s = f"This appears to be {cap}."
        elif style == "Social Media":
            # simple hashtag generator: pick nouns/words (quick naive)
            words = [w.strip(".,!?:;").lower() for w in cap.split() if len(w) > 3]
            tags = "#" + " #".join(words[:3]) if words else ""
            s = f"{cap} {tags}"
        elif style == "Poetic":
            s = cap.replace(",", " —") + "..."
        elif style == "Story":
            s = cap.capitalize() + ". " + "It looks like a scene from a story."
        else:
            s = cap
        styled.append(s)

    # Build HTML (cards)
    card_colors = [("#fff3f2","#b12b2b"),("#f1fff3","#136a1d"),("#eef7ff","#05506e")]
    html = "<div style='display:flex;flex-direction:column;gap:12px;font-family:Inter, Arial, sans-serif'>"
    for i, c in enumerate(styled):
        bg, fg = card_colors[i % len(card_colors)]
        html += f"""
            <div style="padding:14px;border-radius:10px;background:{bg};color:{fg};font-size:16px;box-shadow:0 6px 18px rgba(0,0,0,0.06)">
               <b>📝 Caption {i+1}:</b> {c}
            </div>
        """
    html += "</div>"

    # Compute diversity/quality metric
    dscore = diversity_score(captions)

    # plain text for downloads (English base)
    plain = "\n".join([f"{i+1}. {c}" for i,c in enumerate(styled)])

    return html, plain, dscore

# ======= Save helpers (txt, json, srt) =======
def save_as_files(plain_text, captions_html, lang_code="en"):
    """
    Saves captions to captions.txt, captions.json, captions.srt.
    Returns dict of file paths so Gradio File component can return chosen file.
    """
    if not plain_text:
        return None, None, None

    uid = str(uuid.uuid4())[:8]
    txt_path = f"captions_{uid}.txt"
    json_path = f"captions_{uid}.json"
    srt_path = f"captions_{uid}.srt"

    now = datetime.utcnow().isoformat()
    # TXT
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"Generated Captions ({now})\n\n")
        f.write(plain_text)

    # JSON structure
    items = []
    lines = plain_text.splitlines()
    for i,line in enumerate(lines):
        items.append({"index": i+1, "caption": line.strip()})
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({"generated_at": now, "language": lang_code, "captions": items}, f, ensure_ascii=False, indent=2)

    # SRT (very simple: each caption gets 3 seconds)
    with open(srt_path, "w", encoding="utf-8") as f:
        for i,line in enumerate(lines):
            start = i*3
            end = start + 3
            def fmt(ms):
                h = ms//3600
                m = (ms%3600)//60
                s = ms%60
                return f"{h:02d}:{m:02d}:{s:02d},000"
            f.write(f"{i+1}\n{fmt(start)} --> {fmt(end)}\n{line}\n\n")

    return txt_path, json_path, srt_path

# ======= TTS function (gTTS) =======
def text_to_speech(plain_text, lang="en"):
    if not plain_text:
        return None
    uid = str(uuid.uuid4())[:8]
    mp3_path = f"captions_{uid}.mp3"
    tts = gTTS(plain_text, lang=lang)
    tts.save(mp3_path)
    return mp3_path

# ======= Gradio callbacks: generate, download, translate, tts =======
# hidden_plain stores the latest plain-text captions (English-styled)
def on_generate(image, style, creativity, max_len, num_captions):
    html, plain, dscore = gen_captions_and_plain(image, style, creativity, max_len, num_captions)
    # return HTML for display, plain text for hidden storage, diversity score
    return html, plain, f"{dscore}% diversity"

def on_download(plain_text, lang_select, format_select):
    # translate if needed for file content
    lang_code = lang_select
    if lang_code != "en" and plain_text:
        # translate each line simply (call translate_text)
        lines = plain_text.splitlines()
        translated_lines = []
        for line in lines:
            # naive: remove "1. " prefix if present
            prefix = ""
            if line.strip() and line.strip()[0].isdigit() and "." in line:
                # split
                parts = line.split(".",1)
                if len(parts)>1:
                    prefix = parts[0] + ". "
                    textpart = parts[1].strip()
                else:
                    textpart = line
            else:
                textpart = line
            # do translation
            try:
                trans = translate_text(textpart, lang_code)
            except Exception:
                trans = textpart
            translated_lines.append(prefix + trans)
        target_plain = "\n".join(translated_lines)
    else:
        target_plain = plain_text

    txt_path, json_path, srt_path = save_as_files(target_plain, None, lang_code)
    chosen = format_select
    if chosen == "txt":
        return txt_path
    elif chosen == "json":
        return json_path
    else:
        return srt_path

def on_tts(plain_text, lang_select):
    if not plain_text:
        return None
    # pick first caption line (or whole text)
    # by default, convert whole plain_text
    lang = lang_select
    try:
        mp3 = text_to_speech(plain_text, lang=lang)
        return mp3
    except Exception as e:
        print("TTS error:", e)
        return None

# ======= Build Gradio UI =======
with gr.Blocks() as demo:
    gr.Markdown("# 🖼️ Advanced Caption Studio — BLIP + UX Suite")
    gr.Markdown("Generate multiple captions, tune creativity/length, select style, translate, download & audio.")

    with gr.Row():
        with gr.Column(scale=1):
            img = gr.Image(type="pil", label="📂 Upload Image")
            style = gr.Radio(["Short & Direct","Fun & Creative","Descriptive & Detailed","Social Media","Poetic","Story"],
                             value="Descriptive & Detailed", label="Caption Style")
            creativity = gr.Slider(0.1, 1.5, value=0.8, step=0.05, label="Creativity (temperature)")
            max_len = gr.Slider(10, 60, value=30, step=5, label="Max Caption Length")
            num_caps = gr.Slider(1,5, value=3, step=1, label="Number of captions to generate")
            btn_generate = gr.Button("✨ Generate Captions")
            btn_regen = gr.Button("🔄 Regenerate (same image)")
        with gr.Column(scale=1):
            caption_html = gr.HTML(label="📝 Captions (styled cards)")
            diversity = gr.Textbox(label="Diversity Score", interactive=False)
            # Hidden plain text storage for downloads/TTS
            hidden_plain = gr.Textbox(visible=False)
            gr.Markdown("**Translation & Download**")
            lang = gr.Dropdown(choices=["en","hi","fr","es","de","zh"], value="en", label="File language (translate to)")
            fmt = gr.Dropdown(choices=["txt","json","srt"], value="txt", label="Download format")
            btn_download = gr.Button("⬇️ Download Captions (translated)")
            btn_tts = gr.Button("🔊 Generate & Play Audio (TTS)")
            audio_out = gr.Audio(label="🔊 TTS Output", interactive=False)
            file_out = gr.File(label="📥 Download File")

    # Hook up actions
    btn_generate.click(fn=on_generate, inputs=[img, style, creativity, max_len, num_caps],
                       outputs=[caption_html, hidden_plain, diversity])
    btn_regen.click(fn=on_generate, inputs=[img, style, creativity, max_len, num_caps],
                    outputs=[caption_html, hidden_plain, diversity])

    btn_download.click(fn=on_download, inputs=[hidden_plain, lang, fmt], outputs=file_out)
    btn_tts.click(fn=on_tts, inputs=[hidden_plain, lang], outputs=audio_out)

# Launch app (Colab requires share=True)
demo.launch(share=True, debug=True, theme=gr.themes.Soft())

Found existing installation: click 8.1.7
Uninstalling click-8.1.7:
  Successfully uninstalled click-8.1.7
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.2.0 which is incompatible.
rasterio 1.5.0 requires click!=8.2.*,>=4.0, but you have click 8.2.0 which is incompatible.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d8bb2b58dcbe24ee73.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/tmp/ipykernel_1385/2235727919.py:148: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow().iso

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d8bb2b58dcbe24ee73.gradio.live
